# 02 – Does the model copy the labels?
Now the model is used for the first time. We score the same 600 applicants and compare its decisions
with the labels.

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from common import load_data, predict

df = load_data()
df["label_rejected"] = df.decision == "reject"
df["risk_score"], df["model_rejected"] = predict(df)
df["agree"] = df.label_rejected == df.model_rejected

## 1. Overall agreement

In [2]:
print("Agreement with labels:", round(df.agree.mean(), 4))
pd.crosstab(df.label_rejected, df.model_rejected)

Agreement with labels: 0.9917


model_rejected,False,True
label_rejected,,
False,242,2
True,3,353


## 2. How sure is the model?

In [3]:
sure = ((df.risk_score < 0.01) | (df.risk_score > 0.99)).mean()
print(f"Scores below 0.01 or above 0.99: {sure:.1%}")

Scores below 0.01 or above 0.99: 93.8%


Almost every score is near 0 or near 1. The model behaves like a set of hard yes/no rules
rather than weighing evidence gradually.

## 3. The applicants where the model and labels disagree

In [4]:
df[~df.agree]

,application_id,age,monthly_income,loan_amount,tenure_months,employment_years,credit_score,existing_loans,pincode,decision,label_rejected,risk_score,model_rejected,agree
208,A50209,27,142623,1350000,48,1,664,1,401538,approve,False,0.8967,True,False
211,A50212,59,143163,280000,48,7,710,1,415489,approve,False,0.5645,True,False
238,A50239,61,172126,670000,24,13,705,1,410719,reject,True,0.4997,False,False
374,A50375,59,96758,735000,24,2,748,3,419908,reject,True,0.3970,False,False
509,A50510,54,104813,825000,48,1,714,1,411140,reject,True,0.4508,False,False


**What to notice:** all 5 disagreements have pincodes starting with **40 or 41**. That's a lead for
notebook 06.

## 4. Does the model copy the suspicious groups from notebook 01?

In [5]:
groups = {
    "existing_loans == 0": df.existing_loans == 0,
    "monthly_income >= 3,00,000": df.monthly_income >= 300000,
    "credit_score < 633": df.credit_score < 633,
    "everyone else": (df.existing_loans > 0) & (df.monthly_income < 300000) & (df.credit_score >= 633),
}
table = pd.DataFrame([{"group": name, "applicants": m.sum(),
                       "label_reject_rate": df.label_rejected[m].mean(),
                       "model_reject_rate": df.model_rejected[m].mean(),
                       "agreement": df.agree[m].mean()} for name, m in groups.items()]).round(3)
table

,group,applicants,label_reject_rate,model_reject_rate,agreement
0,existing_loans == 0,108,1.000,1.000,1.000
1,"monthly_income >= 3,00,000",22,1.000,1.000,1.000
2,credit_score < 633,77,1.000,1.000,1.000
3,everyone else,411,0.406,0.404,0.988


**Conclusion:** the model copies the labels, odd rules included (100% agreement on every suspicious
group). Because the validation labels contain the same rules, the high accuracy can't reveal them
(see notebook 09).